# Compute mutual information footprints and expression shifts.

In [9]:
from glob import glob

import numpy as np
import pandas as pd

import itertools

from numba import njit

In [10]:
df_wt = pd.read_csv("../../data/metadata/wt_sequences.csv")[['promoter_seq', 'promoter']]
wt_dict = dict(zip(df_wt['promoter'].values, df_wt['promoter_seq'].values))
df_wt

,promoter_seq,promoter
0,TTTCATCTTTTGTCAACCATTCACAGCGCAAATATACGCCTTTTTT...,rspAp
1,TCCACATTGATTATTTGCACGGCGTCACACTTTGCTATGCCATAGC...,araBp
2,GTGTTGCACCTCCCCAGAGAGCGGCGGATAATGCTGCGAAAAGAAG...,znuCp
3,CCAGTTTCATCATTCCATTTTATTTTGCGAGCGAGCGCACACTTGT...,xylAp
4,AATTGCGCAACAAAAGTAAGATCTCGGTCATAAATCAAGAAATAAA...,xylFp
...,...,...
112,TGGTGCTGGTTATCAGCACTGAAGGCGACACCGACGTGAAGCACTA...,ygeY_predicted
113,CGCTGGACGAAATCCGCGCCCTGCCTGCAGTACAGAAAGCTAACGC...,hyuA_predicted
114,TCGGACATGTTTGAAGAGATGAAATTTGCCTTCTTTAAACATCGCG...,ygfM_xdhD_predicted
115,AGTGCCAGTTTTTGCGAGGAAGGGGAGTTAGAGACAGGAATTGCGG...,ygfS_ygfT_predicted


In [11]:
df_map = pd.read_csv("../../data/mapping/mapped_barcodes_filtered.csv")

In [12]:
files = np.hstack([glob("../../data/extracted_barcodes/*DNA*.txt")])
inds = np.sort([x.split('/')[-1].split('_')[0] for x in files])
inds = [file for file in inds if not np.any([y == file for y in  ['14-3', '15-1', '19-1', '22-1', '25-2', '38-2','4-1', '31-2', '20-2', '13-2', '26-1', '26-2']])]
inds[0:5]

[np.str_('1-1'),
 np.str_('1-2'),
 np.str_('10-1'),
 np.str_('10-2'),
 np.str_('11-1')]

## Compute mutual information

In [18]:
@njit
def compare_string_bytes(s1, s2):
    # Ensure both strings are of equal length.
    n = len(s1)
    if n != len(s2):
        raise ValueError("Strings must have the same length")
    result = np.empty(n, dtype=np.int64)
    for i in range(n):
        result[i] = 1 if s1[i] == s2[i] else 0
    return result


def clog(x, y, z):
    if x == 0 or y == 0 or z==0:
        return 0
    else:
        return x * np.log2(x / (y * z))

norm = False

for ind in inds:
    print(ind)
    file_DNA = glob(f"../../data/extracted_barcodes/{ind}*DNA*.txt")[0]
    df_DNA = pd.read_csv(
        file_DNA, 
        names=['ct_0', 'barcode'], sep="\\s+")
    file_RNA = glob(f"../../data/extracted_barcodes/{ind}*RNA*.txt")[0]
    df_RNA = pd.read_csv(
        file_RNA, 
        names=['ct_1', 'barcode'], sep="\\s+")
    
    df_counts = df_DNA.merge(df_RNA, on="barcode", how='outer').fillna(0)
    df_counts = df_counts.merge(df_map, on="barcode", how='inner')
    
    df_counts.head()
    
    df_out = pd.DataFrame()
    for promoter, gdf in df_counts.groupby('promoter'):
        if promoter in ['galEp', 'ybeDp2']:
            continue
        # find where the mutations are
        wt_seq = wt_dict[promoter]
        is_wt = np.vstack([np.array(compare_string_bytes(wt_seq.encode("ascii"), x.encode("ascii"))) for x in gdf['promoter_variant'].values])
        index = (gdf['ct_0'] > 0) & (gdf['ct_1'] > 0)
        if norm:
            x = (gdf['ct_0'].values[index]) / (gdf['ct_0'].values[index])
            y = (gdf['ct_1'].values[index]) / (gdf['ct_0'].values[index])
        else:
            x = gdf['ct_0'].values[index]
            y = gdf['ct_1'].values[index]
        # compute relative counts
        p = np.zeros([160, 2, 2])
        p [:, :, 0] = np.matmul(is_wt[index].T, np.vstack([x, y]).T)
        p [:, :, 1] = np.matmul((1-is_wt[index]).T, np.vstack([x, y]).T)
        p = p / (np.sum(x + y))
    
        mut_info = [sum([clog(p[i, j, k], np.sum(p[i, :, k]), np.sum(p[i, j, :])) for j in range(2) for k in range(2)]) for i in range(160)]
        df_out = pd.concat([df_out, pd.DataFrame({"promoter": promoter, "mut_info": mut_info, "pos": np.arange(-115, 45)})])
    df_out.to_csv(f"footprints/{ind}_footprints.csv")
    print(f"{ind} done")

1-1
1-1 done
1-2
1-2 done
10-1
10-1 done
10-2
10-2 done
11-1
11-1 done
11-2
11-2 done
12-1
12-1 done
12-2
12-2 done
13-1
13-1 done
13-3
13-3 done
14-1
14-1 done
14-2
14-2 done
15-2
15-2 done
15-3
15-3 done
16-1
16-1 done
16-2
16-2 done
17-1
17-1 done
17-2
17-2 done
18-1
18-1 done
18-2
18-2 done
19-2
19-2 done
19-3
19-3 done
2-1
2-1 done
2-2
2-2 done
20-1
20-1 done
20-3
20-3 done
21-1
21-1 done
21-2
21-2 done
22-2
22-2 done
22-3
22-3 done
23-1
23-1 done
23-2
23-2 done
24-1
24-1 done
24-2
24-2 done
25-1
25-1 done
25-3
25-3 done
28-1
28-1 done
28-2
28-2 done
29-1
29-1 done
29-2
29-2 done
3-1
3-1 done
3-2
3-2 done
30-1
30-1 done
30-2
30-2 done
31-1
31-1 done
31-3
31-3 done
32-1
32-1 done
32-2
32-2 done
33-1
33-1 done
33-2
33-2 done
34-1
34-1 done
34-2
34-2 done
35-1
35-1 done
35-2
35-2 done
36-1
36-1 done
36-2
36-2 done
37-1
37-1 done
37-2
37-2 done
38-1
38-1 done
38-3
38-3 done
39-1
39-1 done
39-2
39-2 done
4-2
4-2 done
4-3
4-3 done
40-1
40-1 done
40-2
40-2 done
40-3
40-3 done
41-1
41-1 d

## Compute expression shifts

In [19]:
# Precompute the mapping outside the function
DNA_MAPPING = np.zeros(256, dtype=np.uint8)
DNA_MAPPING[ord('A')] = 0
DNA_MAPPING[ord('C')] = 1
DNA_MAPPING[ord('G')] = 2
DNA_MAPPING[ord('T')] = 3

def dna_to_int(seq, mapping=DNA_MAPPING):
    # Convert the sequence to a numpy array of its ASCII codes.
    arr = np.frombuffer(seq.encode('ascii'), dtype=np.uint8)
    return mapping[arr]


def batch_dna_to_int(sequences, mapping=DNA_MAPPING):
    # Assume all sequences are the same length.
    n = len(sequences[0])
    # Create a 2D array of shape (num_sequences, n)
    seq_arr = np.empty((len(sequences), n), dtype=np.uint8)
    for i, seq in enumerate(sequences):
        seq_arr[i, :] = np.frombuffer(seq.encode('ascii'), dtype=np.uint8)
    return mapping[seq_arr]


@njit
def accumulate_shifts(int_seq, is_mut, exp_diff, ex_shift_arr):
    num_variants, seq_len = int_seq.shape
    for i in range(num_variants):
        for j in range(seq_len):
            base = int_seq[i, j]
            ex_shift_arr[j, base] += exp_diff[i] * is_mut[i, j]


positions = list(itertools.product(range(-115, 45), range(1, 5)))

for ind in inds:
    # import DNA file
    file_DNA = glob(f"../../data/extracted_barcodes/{ind}*DNA*.txt")[0]
    df_DNA = pd.read_csv(
        file_DNA, 
        names=['ct_0', 'barcode'], sep="\\s+")
    file_RNA = glob(f"../../data/extracted_barcodes/{ind}*RNA*.txt")[0]
    df_RNA = pd.read_csv(
        file_RNA, 
        names=['ct_1', 'barcode'], sep="\\s+")

    # merge files
    df_counts = df_DNA.merge(df_RNA, on="barcode", how='outer').fillna(0)
    df_counts = df_counts.merge(df_map, on="barcode", how='inner')
    
    # iterate through promoters
    df_out_arr = [] 
    for promoter, gdf in df_counts.groupby('promoter'):
        
        # find where the mutations are
        wt_seq = wt_dict[promoter]
        is_mut = 1 - np.vstack([np.array(compare_string_bytes(wt_seq.encode("ascii"), x.encode("ascii"))) for x in gdf['promoter_variant'].values])
        int_seq = batch_dna_to_int(gdf['promoter_variant'].values)

        # compute relative counts and difference to mean
        relative_counts = (gdf['ct_1'].values + 1) / (gdf['ct_0'].values + 1)
        mean_rel_counts = np.mean(relative_counts)
        exp_diff = relative_counts - mean_rel_counts
        
        # initialize expression shift array
        ex_shift_arr = np.zeros((160, 4))

        # fill array
        accumulate_shifts(int_seq, is_mut, exp_diff, ex_shift_arr)

        # get wild type base column for dataframe
        ind_wt = list(itertools.product(list(wt_seq), range(1, 5)))

        # fill dataframe 
        df_out_arr.append(pd.DataFrame({"promoter": promoter, 
                                          "expression_shift": np.ravel(ex_shift_arr).T, 
                                          "pos":[t[0] for t in positions],
                                          "base":[t[1] for t in positions],
                                          "wt_base":[t[0] for t in ind_wt]
                                         }))

    pd.concat(df_out_arr).to_csv(f"expression_shifts/{ind}_exshifts.csv")
    print(f"{ind} done")


KeyError: 'galEp'